# **Support Vector Machine**

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [6]:
train.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S


In [7]:
test.head(1)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q


# Feature Engineering
# dataset['FamilySize'] = dataset['SibSp'] + dataset['Parch']
# dataset['Title'] = dataset['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [8]:
print(train.shape, test.shape)

(891, 12) (418, 11)


In [9]:
train['data'] = 'train'
test['data'] = 'test'

In [7]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
data             0
dtype: int64

In [8]:
test.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
data             0
dtype: int64

In [9]:
test_passenger_id = test['PassengerId']

In [10]:
dataset = pd.concat([train, test], axis=0)

In [11]:
dataset.shape

(1309, 13)

In [12]:
dataset.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,data
413,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,test
414,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,test
415,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,test
416,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S,test
417,1309,NaN,3,"Peter, Master. Michael J",male,NaN,1,1,2668,22.3583,NaN,C,test


In [13]:
dataset.isnull().sum()/len(dataset)*100

PassengerId     0.000000
Survived       31.932773
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            20.091673
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.076394
Cabin          77.463713
Embarked        0.152788
data            0.000000
dtype: float64

In [14]:
dataset = dataset.drop(['PassengerId','Name', 'Ticket','Cabin'], axis=1)

In [15]:
dataset.isnull().sum()

Survived    418
Pclass        0
Sex           0
Age         263
SibSp         0
Parch         0
Fare          1
Embarked      2
data          0
dtype: int64

In [16]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())

In [17]:
dataset['Embarked'].value_counts()

Embarked
S    914
C    270
Q    123
Name: count, dtype: int64

In [18]:
dataset['Embarked'] = dataset['Embarked'].fillna('S')

In [19]:
# dataset['Fare'] = dataset['Fare'].fillna(dataset['Fare'].median())
dataset['Fare'] = dataset.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

In [20]:
dataset['Fare'].nunique()

281

In [21]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1309 entries, 0 to 417
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    float64
 1   Pclass    1309 non-null   int64  
 2   Sex       1309 non-null   object 
 3   Age       1309 non-null   float64
 4   SibSp     1309 non-null   int64  
 5   Parch     1309 non-null   int64  
 6   Fare      1309 non-null   float64
 7   Embarked  1309 non-null   object 
 8   data      1309 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 102.3+ KB


In [22]:
dataset = pd.get_dummies(dataset, columns=['Sex', 'Embarked','Pclass'], drop_first=True)

In [23]:
dataset.head()

,Survived,Age,SibSp,Parch,Fare,data,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,0.0,22.0,1,0,7.2500,train,True,False,True,False,True
1,1.0,38.0,1,0,71.2833,train,False,False,False,False,False
2,1.0,26.0,0,0,7.9250,train,False,False,True,False,True
3,1.0,35.0,1,0,53.1000,train,False,False,True,False,False
4,0.0,35.0,0,0,8.0500,train,True,False,True,False,True


In [24]:
train = dataset[dataset['data']=='train']
test = dataset[dataset['data']=='test']

In [25]:
print(train.shape, test.shape)

(891, 11) (418, 11)


In [26]:
test.columns

Index(['Survived', 'Age', 'SibSp', 'Parch', 'Fare', 'data', 'Sex_male',
       'Embarked_Q', 'Embarked_S', 'Pclass_2', 'Pclass_3'],
      dtype='object')

In [27]:
train.columns

Index(['Survived', 'Age', 'SibSp', 'Parch', 'Fare', 'data', 'Sex_male',
       'Embarked_Q', 'Embarked_S', 'Pclass_2', 'Pclass_3'],
      dtype='object')

In [28]:
test[['Survived']]

,Survived
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
413,NaN
414,NaN
415,NaN
416,NaN


In [29]:
train.head()

,Survived,Age,SibSp,Parch,Fare,data,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,0.0,22.0,1,0,7.2500,train,True,False,True,False,True
1,1.0,38.0,1,0,71.2833,train,False,False,False,False,False
2,1.0,26.0,0,0,7.9250,train,False,False,True,False,True
3,1.0,35.0,1,0,53.1000,train,False,False,True,False,False
4,0.0,35.0,0,0,8.0500,train,True,False,True,False,True


# Split the data into x and y

In [30]:
x = train.drop(['Survived', 'data'], axis=1)
y = train['Survived']

In [31]:
x.head()

,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,22.0,1,0,7.2500,True,False,True,False,True
1,38.0,1,0,71.2833,False,False,False,False,False
2,26.0,0,0,7.9250,False,False,True,False,True
3,35.0,1,0,53.1000,False,False,True,False,False
4,35.0,0,0,8.0500,True,False,True,False,True


In [32]:
x.shape

(891, 9)

In [33]:
y.head()

0    0.0
1    1.0
2    1.0
3    1.0
4    0.0
Name: Survived, dtype: float64

In [34]:
y.value_counts()

Survived
0.0    549
1.0    342
Name: count, dtype: int64

In [35]:
test.head()

,Survived,Age,SibSp,Parch,Fare,data,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,NaN,34.5,0,0,7.8292,test,True,True,False,False,True
1,NaN,47.0,1,0,7.0000,test,False,False,True,False,True
2,NaN,62.0,0,0,9.6875,test,True,True,False,True,False
3,NaN,27.0,0,0,8.6625,test,True,False,True,False,True
4,NaN,22.0,1,1,12.2875,test,False,False,True,False,True


In [36]:
test = test.drop(['Survived', 'data'], axis=1)

In [37]:
test.head()

,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,34.5,0,0,7.8292,True,True,False,False,True
1,47.0,1,0,7.0000,False,False,True,False,True
2,62.0,0,0,9.6875,True,True,False,True,False
3,27.0,0,0,8.6625,True,False,True,False,True
4,22.0,1,1,12.2875,False,False,True,False,True


In [38]:
test.shape

(418, 9)

# split the training data into train and test

In [39]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2,stratify=y,random_state=500)

In [40]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

test = scaler.transform(test)

# Support Vector Machine

In [41]:
from sklearn.svm import SVC

#kernel - linear
svm_linear = SVC(kernel='linear')

svm_linear.fit(x_train, y_train)


,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [42]:
y_pred_train_linear = svm_linear.predict(x_train)
y_pred_test_linear = svm_linear.predict(x_test)

# SVC(kernel='linear')  is dual form as it can use kernal tricks.
# LinearSVC()  use primal form which can not use kernal tricks
# Why Dual is Preferred in SVC?

- Because dual complexity depends on:
- number of samples
- number of samples
- Primal complexity depends on:
- number of features
- number of features
- So: If samples small → dual good
- If features large → primal better

| Model         | Solves | Kernel Support   | When to Use                  |
| ------------- | ------ | ---------------- | ---------------------------- |
| `SVC()`       | Dual   | Yes              | Small/medium data, nonlinear |
| `LinearSVC()` | Primal | No (linear only) | Large datasets               |


In [43]:
print(y_train.shape)

(712,)


In [44]:
#kernel - sigmoid
svm_sigmoid = SVC(kernel='sigmoid')
svm_sigmoid.fit(x_train, y_train)
y_pred_train_sigmoid = svm_sigmoid.predict(x_train)
y_pred_test_sigmoid = svm_sigmoid.predict(x_test)



In [45]:
#kernel - poly
svm_poly = SVC(kernel='poly')
svm_poly.fit(x_train, y_train)
y_pred_train_poly = svm_poly.predict(x_train)
y_pred_test_poly = svm_poly.predict(x_test)


In [46]:
#kernel - rbf
svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(x_train, y_train)
y_pred_train_rbf = svm_rbf.predict(x_train)
y_pred_test_rbf = svm_rbf.predict(x_test)

In [47]:
y_pred_test_rbf

array([0., 0., 1., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 0., 1., 1.,
       0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 1.,
       0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,
       0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 1., 0.,
       0., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0.,
       1., 1., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 1., 0., 0., 1., 0.,
       1., 0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1.,
       0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1.,
       1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 1., 1., 0.,
       0., 0., 0., 1., 0., 1., 0., 1., 1.])

# Evulation 

In [48]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [49]:
print("Training Accuracy - Linear :", accuracy_score(y_train, y_pred_train_linear))
print("*************"*5)
print("Test Accuracy - Linear :", accuracy_score(y_test, y_pred_test_linear))
print("*************"*5)


Training Accuracy - Linear : 0.7752808988764045
*****************************************************************
Test Accuracy - Linear : 0.8324022346368715
*****************************************************************


In [50]:
print("Training Accuracy - sigmoid :", accuracy_score(y_train, y_pred_train_sigmoid))
print("*************"*5)
print("Test Accuracy - sigmoid :", accuracy_score(y_test, y_pred_test_sigmoid))
print("*************"*5)


Training Accuracy - sigmoid : 0.6460674157303371
*****************************************************************
Test Accuracy - sigmoid : 0.6815642458100558
*****************************************************************


In [51]:
print("Training Accuracy - poly :", accuracy_score(y_train, y_pred_train_poly))
print("*************"*5)
print("Test Accuracy - poly :", accuracy_score(y_test, y_pred_test_poly))
print("*************"*5)


Training Accuracy - poly : 0.8286516853932584
*****************************************************************
Test Accuracy - poly : 0.8324022346368715
*****************************************************************


In [52]:
print("Training Accuracy - rbf :", accuracy_score(y_train, y_pred_train_rbf))
print("*************"*5)
print("Test Accuracy - rbf :", accuracy_score(y_test, y_pred_test_rbf))

Training Accuracy - rbf : 0.8342696629213483
*****************************************************************
Test Accuracy - rbf : 0.8547486033519553


In [53]:
print("Training Accuracy - Linear :", classification_report(y_train, y_pred_train_linear))
print("*************"*5)
print("Test Accuracy - Linear :", classification_report(y_test, y_pred_test_linear))


Training Accuracy - Linear :               precision    recall  f1-score   support

         0.0       0.79      0.86      0.83       439
         1.0       0.74      0.64      0.69       273

    accuracy                           0.78       712
   macro avg       0.77      0.75      0.76       712
weighted avg       0.77      0.78      0.77       712

*****************************************************************
Test Accuracy - Linear :               precision    recall  f1-score   support

         0.0       0.90      0.82      0.86       110
         1.0       0.75      0.86      0.80        69

    accuracy                           0.83       179
   macro avg       0.82      0.84      0.83       179
weighted avg       0.84      0.83      0.83       179



# Cross validation method

In [54]:
from sklearn.model_selection import cross_val_score
train_accuracy = cross_val_score(svm_linear, x_train, y_train, cv=10)

print("Training accuracy :", train_accuracy)
print("************"*5)
print("Training Mean Accuracy :", train_accuracy.mean())
print("************"*5)
print("Training Max Accuracy :", train_accuracy.max())
print()
print("*************"*5)
print("Test Accuracy - Linear :", accuracy_score(y_test, y_pred_test_linear))



Training accuracy : [0.79166667 0.86111111 0.71830986 0.76056338 0.73239437 0.8028169
 0.74647887 0.74647887 0.78873239 0.77464789]
************************************************************
Training Mean Accuracy : 0.7723200312989046
************************************************************
Training Max Accuracy : 0.8611111111111112

*****************************************************************
Test Accuracy - Linear : 0.8324022346368715


In [65]:
from sklearn.model_selection import cross_val_score
train_accuracy = cross_val_score(svm_rbf, x_train, y_train, cv=10)

print("Training accuracy :", train_accuracy)
print("************"*5)
print("Training Mean Accuracy :", train_accuracy.mean())
print("************"*5)
print("Training Max Accuracy :", train_accuracy.max())
print()
print("*************"*5)
print("Test Accuracy - RBF :", accuracy_score(y_test, y_pred_test_rbf))

Training accuracy : [0.86111111 0.90277778 0.74647887 0.87323944 0.81690141 0.81690141
 0.83098592 0.76056338 0.81690141 0.8028169 ]
************************************************************
Training Mean Accuracy : 0.8228677621283256
************************************************************
Training Max Accuracy : 0.9027777777777778

*****************************************************************
Test Accuracy - RBF : 0.8547486033519553


# Grid Search CV

In [56]:
from sklearn.model_selection import GridSearchCV

In [57]:
SVC()

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [58]:
param_grid = {'C':[0.1,1,2,10,100], 'gamma':[1,0.1,0.01,0.001]}

grid = GridSearchCV(SVC(),param_grid=param_grid,cv = 5, refit=True)
grid.fit(x_train, y_train)
grid_predict = grid.predict(x_test)
print(accuracy_score(y_test, grid_predict))


0.8491620111731844


In [66]:
final_predictions = svm_rbf.predict(test)

In [67]:
final_predictions

array([0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0.,
       0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 0.,
       0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 1., 0.,
       0., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0.,
       0., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0.,
       1., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 1., 0., 1.,
       0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 1., 0., 1., 1.

In [68]:
np.unique(final_predictions, return_counts=True)

(array([0., 1.]), array([292, 126], dtype=int64))

In [69]:
len(final_predictions)

418

In [71]:
test_df = pd.DataFrame({"PassengerId": test_passenger_id,"Survived": final_predictions
})

test_df.tail()

,PassengerId,Survived
413,1305,0.0
414,1306,1.0
415,1307,0.0
416,1308,0.0
417,1309,0.0


In [64]:
#test_df.to_csv("submission.csv", index=False)